# Évaluation d'Impact — Store Matching & Validation Statistique

Ce notebook met en œuvre une méthode d'appariement statistique pour mesurer l'impact réel de l'activation produit sur le chiffre d'affaires.

**Objectif :** Isoler l'effet de la stratégie marketing des variables externes (saisonnalité, tendances locales) en sélectionnant des magasins témoins comparables à chaque magasin test.

**Démarche :**
1. Sélection des bassins de contrôle par filtrage de profil (volume de transactions × panier moyen)
2. Affinage via corrélation de Pearson sur l'historique des ventes pré-test
3. Mesure du lift de CA et validation statistique (test de Welch)

## 1. Imports & Connexion

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from snowflake.snowpark.context import get_active_session

# Initialisation de la session Snowflake
session = get_active_session()

## 2. Chargement & Préparation des Données

In [ ]:
# Chargement de la table de faits depuis la couche Gold
df = session.table("GOLD.FACT_SALES").to_pandas()

# Conversion de la date de transaction au format datetime
df['TXN_DATE'] = pd.to_datetime(df['TXN_DATE'])

In [ ]:
# Identification et exclusion des magasins avec des mois manquants
# Un magasin avec moins de 12 mois de données biaiserait les comparaisons de saisonnalité
df_store_verification = df.groupby(['STORE_ID'], as_index=False).agg(
    NBR_OF_MONTH=('TXN_MONTH_NAME', 'nunique')
).sort_values(by='NBR_OF_MONTH', ascending=True)

magasins_incomplets = df_store_verification[
    df_store_verification['NBR_OF_MONTH'] < 12
]['STORE_ID'].tolist()

df = df[~df['STORE_ID'].isin(magasins_incomplets)]

print(f"{len(magasins_incomplets)} magasin(s) exclu(s) pour données incomplètes.")

## 3. Analyse de Saisonnalité

Avant de sélectionner les magasins témoins, on visualise la saisonnalité globale des ventes.
Cela permet de valider que l'historique pré-test est suffisamment stable pour servir de référence.

In [ ]:
# Agrégation des ventes par jour
df_quotidien = df.groupby('TXN_DATE')['SALES_AMNT'].sum().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    df_quotidien['TXN_DATE'],
    df_quotidien['SALES_AMNT'],
    color='#004a99',
    linewidth=1,
    alpha=0.8
)

# Style épuré
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
ax.grid(axis='y', linestyle='-', alpha=0.1)
ax.set_title("Saisonnalité Quotidienne des Ventes",
             loc='left', fontsize=14, fontweight='bold', color='#333333', pad=20)
ax.set_ylabel("Ventes ($)", fontsize=10, color='#666666')

plt.tight_layout()
plt.show()

## 4. Sélection des Magasins Témoins (Store Matching)

### Étape 1 — Filtrage par profil
On isole les données pré-test (avant février 2019) et on agrège les KPIs par magasin.
Chaque magasin test est associé à un bassin de contrôle constitué de magasins au profil similaire
(volume de transactions × panier moyen).

In [ ]:
# Isolation de la période pré-test (juillet 2018 — janvier 2019)
df_pre_test = df[df['TXN_DATE'] < '2019-02-01']

In [ ]:
# Agrégation des KPIs par magasin sur la période pré-test
df_agg_store = df_pre_test.groupby(['STORE_ID'], as_index=False).agg(
    NBR_OF_CUST=('TXN_ID', 'nunique'),
    NBR_OF_TXN=('CUSTOMER_ID', 'nunique'),
    SALES_AMNT=('SALES_AMNT', 'sum'),
    SALES_QTY=('SALES_QTY', 'sum')
)

# Calcul du panier moyen — indicateur clé pour l'appariement des profils
df_agg_store['PANIER_MOYEN'] = df_agg_store['SALES_AMNT'] / df_agg_store['NBR_OF_TXN']

In [ ]:
# Définition des bassins de contrôle par filtrage sur le volume de transactions et le panier moyen
pool77 = df_agg_store[
    (df_agg_store["NBR_OF_TXN"] > 150) &
    (df_agg_store["NBR_OF_TXN"] < 300) &
    (df_agg_store["PANIER_MOYEN"] < 10) &
    (df_agg_store["PANIER_MOYEN"] > 4)
]['STORE_ID'].tolist()

pool86 = df_agg_store[
    (df_agg_store["NBR_OF_TXN"] > 200) &
    (df_agg_store["NBR_OF_TXN"] < 300) &
    (df_agg_store["PANIER_MOYEN"] < 30) &
    (df_agg_store["PANIER_MOYEN"] > 20)
]['STORE_ID'].tolist()

pool88 = df_agg_store[
    (df_agg_store["NBR_OF_TXN"] > 300) &
    (df_agg_store["NBR_OF_TXN"] < 380) &
    (df_agg_store["PANIER_MOYEN"] < 25) &
    (df_agg_store["PANIER_MOYEN"] > 18)
]['STORE_ID'].tolist()

# Mapping de chaque magasin candidat à son bassin de contrôle
group_mapping = {}
for store_id in pool77:
    group_mapping[store_id] = 77
for store_id in pool86:
    group_mapping[store_id] = 86
for store_id in pool88:
    group_mapping[store_id] = 88

df_agg_store['STORE_GROUP'] = df_agg_store['STORE_ID'].map(group_mapping).fillna("Hors Pool")

print(f"Bassin 77 : {len(pool77)} magasins candidats")
print(f"Bassin 86 : {len(pool86)} magasins candidats")
print(f"Bassin 88 : {len(pool88)} magasins candidats")

In [ ]:
# Visualisation des bassins de contrôle — Volume de Transactions × Panier Moyen
# Chaque couleur représente un bassin, les gros points foncés sont les magasins test

COLOR_MAP = {
    77:         "#b0c4d8",  # Bleu clair
    86:         "#7a9ab5",  # Bleu moyen
    88:         "#d4b896",  # Beige
    "Hors Pool":"#e0e0e0"   # Gris neutre
}

fig, ax = plt.subplots(figsize=(10, 7))

for group, subset in df_agg_store.groupby('STORE_GROUP'):
    is_test = subset['STORE_ID'].isin([77, 86, 88])

    # Magasins candidats (petits points)
    ax.scatter(
        subset[~is_test]['NBR_OF_TXN'],
        subset[~is_test]['PANIER_MOYEN'],
        color=COLOR_MAP.get(group, "#e0e0e0"),
        alpha=0.5,
        s=40,
        label=f"Bassin de contrôle {group}" if group != "Hors Pool" else "Hors segments d'étude"
    )

    # Magasins test (gros points foncés)
    ax.scatter(
        subset[is_test]['NBR_OF_TXN'],
        subset[is_test]['PANIER_MOYEN'],
        color="#1a2e40",
        s=160,
        zorder=5
    )

    # Labels des magasins test
    for _, row in subset[is_test].iterrows():
        ax.annotate(
            f"TEST {int(row['STORE_ID'])}",
            (row['NBR_OF_TXN'], row['PANIER_MOYEN']),
            textcoords="offset points",
            xytext=(8, 4),
            fontsize=9,
            fontweight='bold',
            color='#1a2e40'
        )

# Style épuré
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
ax.grid(linestyle='-', alpha=0.1)
ax.set_title("Stratégie d'Appariement : Segmentation des Bassins de Contrôle",
             loc='left', fontsize=13, fontweight='bold', color='#333333', pad=20)
ax.set_xlabel("Volume de Transactions", fontsize=10, color='#666666')
ax.set_ylabel("Valeur du Panier Moyen ($)", fontsize=10, color='#666666')
ax.legend(frameon=False, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

### Étape 2 — Affinage par corrélation de Pearson

Au sein de chaque bassin, on calcule la corrélation de Pearson entre l'historique mensuel
des ventes du magasin test et celui de chaque candidat.
Le magasin témoin retenu est celui dont la saisonnalité est la plus proche du magasin test.

In [ ]:
# Agrégation du CA mensuel par magasin — base pour le calcul de corrélation
df_monthly_agg = df.groupby(['STORE_ID', 'TXN_MONTH_NUMBER']).agg(
    SALES_AMNT=('SALES_AMNT', 'sum')
).reset_index()

# Pivot : chaque colonne = un magasin, chaque ligne = un mois
df_pivot_monthly = df_monthly_agg.pivot(
    index='TXN_MONTH_NUMBER',
    columns='STORE_ID',
    values='SALES_AMNT'
).fillna(0)

# Filtrage sur la période pré-test uniquement (mois 1 à 8)
# La corrélation est calculée sur cet historique pour ne pas être
# influencée par la période de test
df_pivot_monthly_pre_test = df_pivot_monthly[df_pivot_monthly.index <= 8]

In [ ]:
# Calcul de la corrélation de Pearson entre chaque magasin test et les candidats de son bassin
magasins_test = [77, 86, 88]

for store_test in magasins_test:
    ids_bassin = df_agg_store[
        df_agg_store['STORE_GROUP'] == store_test
    ]['STORE_ID'].unique()

    # On s'assure que les candidats sont présents dans le pivot
    ids_valides = [s for s in ids_bassin if s in df_pivot_monthly_pre_test.columns]

    correlations = (
        df_pivot_monthly_pre_test[ids_valides]
        .corr()[store_test]
        .sort_values(ascending=False)
    )

    print(f"\nTop magasins contrôles pour le Store Test {store_test} :")
    print(correlations.head(4))

In [ ]:
# Sélection finale des paires test / contrôle
# Pour chaque magasin test, on retient le magasin contrôle le plus corrélé
# Format : liste de tuples (store_test, [stores_controle])
paires = [
    (77, [233]),
    (86, [155]),
    (88, [91])
]

## 5. Visualisation — Performance Test vs Contrôle

On compare l'évolution mensuelle du CA de chaque magasin test avec son magasin contrôle.
La zone grisée représente la période d'activation (février à avril 2019).

In [ ]:
# Paramètres de la période de test
DEBUT_TEST = 8
FIN_TEST   = 10

# Palette sobre
COLOR_TEST = "#004a99"  # Bleu profond
COLOR_CTRL = "#95a5a6"  # Gris neutre
COLOR_SPAN = "#f4f4f4"  # Zone de test très claire

fig, axes = plt.subplots(3, 1, figsize=(10, 12), sharex=True)

for i, (store_test, stores_ctrl) in enumerate(paires):
    # Données du magasin test
    df_test = df_monthly_agg[df_monthly_agg['STORE_ID'] == store_test].copy()

    # Moyenne mensuelle du magasin contrôle
    df_ctrl_avg = (
        df_monthly_agg[df_monthly_agg['STORE_ID'].isin(stores_ctrl)]
        .groupby('TXN_MONTH_NUMBER')['SALES_AMNT']
        .mean()
        .reset_index()
    )

    # Tracé contrôle
    axes[i].plot(
        df_ctrl_avg['TXN_MONTH_NUMBER'],
        df_ctrl_avg['SALES_AMNT'],
        color=COLOR_CTRL,
        label='Moyenne Contrôles (Stable)',
        linestyle='--',
        linewidth=1.5,
        alpha=0.8
    )

    # Tracé magasin test
    axes[i].plot(
        df_test['TXN_MONTH_NUMBER'],
        df_test['SALES_AMNT'],
        color=COLOR_TEST,
        label=f'Magasin Test {store_test}',
        marker='o',
        markersize=5,
        linewidth=2
    )

    # Zone de test
    axes[i].axvspan(DEBUT_TEST, FIN_TEST, color=COLOR_SPAN, label='Période de Test', zorder=0)

    # Style épuré
    axes[i].set_title(
        f"Impact de l'Opération : Analyse du Lift (Magasin {store_test})",
        loc='left', fontsize=11, color='#333333', pad=10
    )
    axes[i].spines['top'].set_visible(False)
    axes[i].spines['right'].set_visible(False)
    axes[i].spines['left'].set_color('#cccccc')
    axes[i].spines['bottom'].set_color('#cccccc')
    axes[i].grid(axis='y', linestyle='-', alpha=0.2)
    axes[i].set_ylabel("CA Mensuel ($)", fontsize=9, color='#666666')

    if i == 0:
        axes[i].legend(frameon=False, loc='upper left', fontsize=9)

plt.xlabel("Mois", fontsize=10, color='#666666')
plt.tight_layout(pad=3.0)
plt.show()

## 6. Calcul du Lift & Validation Statistique

### Méthodologie — Difference-in-Differences (DiD)
- On mesure l'évolution du CA du magasin test entre pré-test et test
- On soustrait l'évolution observée sur le magasin contrôle pour neutraliser les effets externes
- La significativité est évaluée par un **test t de Welch** (adapté aux petits échantillons avec variances potentiellement inégales)

In [ ]:
# Définition des périodes de référence
MOIS_PRE_TEST = [3, 4, 5]   # Mois de référence pré-test
MOIS_TEST     = [8, 9, 10]  # Période d'activation (fév — avr 2019)

resultats_finaux = []

for store_test, stores_ctrl in paires:

    # --- A. Extraction des données par période ---
    df_t = df_monthly_agg[df_monthly_agg['STORE_ID'] == store_test]
    df_c = df_monthly_agg[df_monthly_agg['STORE_ID'].isin(stores_ctrl)]

    # Moyennes de CA pré-test et durant le test
    val_pre_t  = df_t[df_t['TXN_MONTH_NUMBER'].isin(MOIS_PRE_TEST)]['SALES_AMNT'].mean()
    val_post_t = df_t[df_t['TXN_MONTH_NUMBER'].isin(MOIS_TEST)]['SALES_AMNT'].mean()
    val_pre_c  = df_c[df_c['TXN_MONTH_NUMBER'].isin(MOIS_PRE_TEST)]['SALES_AMNT'].mean()
    val_post_c = df_c[df_c['TXN_MONTH_NUMBER'].isin(MOIS_TEST)]['SALES_AMNT'].mean()

    # --- B. Calcul Difference-in-Differences ---
    delta_t = (val_post_t / val_pre_t) - 1
    delta_c = (val_post_c / val_pre_c) - 1
    lift    = delta_t - delta_c

    # CA incrémental généré sur la période de test
    ca_incremental = lift * val_pre_t * len(MOIS_TEST)

    # --- C. Test de significativité (Welch's t-test) ---
    v_test = df_t[df_t['TXN_MONTH_NUMBER'].isin(MOIS_TEST)]['SALES_AMNT']
    v_ctrl = df_c[df_c['TXN_MONTH_NUMBER'].isin(MOIS_TEST)]['SALES_AMNT']
    _, p_val = stats.ttest_ind(v_test, v_ctrl, equal_var=False)

    # --- D. Compilation ---
    resultats_finaux.append({
        'Magasin Test':    f"Store {store_test}",
        'Impact (Lift %)': f"{lift * 100:+.2f}%",
        'P-Value':         round(p_val, 4),
        'Confiance':       f"{(1 - p_val) * 100:.1f}%",
        'Statut':          "Validé ✅" if p_val < 0.05 else "Non significatif"
    })

# Affichage du tableau de synthèse
df_summary = pd.DataFrame(resultats_finaux)
print(df_summary.to_string(index=False))